# Extracting data

### Reading MIBOR-OIS rates

In [2]:
import pandas as pd
import requests
import json

In [3]:
mibor_ois_url = "https://www.fbil.org.in/wasdm/miborois/fetchcommongraph/3M"

r1 = requests.get(mibor_ois_url)

r1.raise_for_status()

data1 = r1.json()

mibor_ois_df = pd.DataFrame(data1)

mibor_ois_df = pd.concat(
    [mibor_ois_df.drop(columns="benchMarkTrendData"),
     mibor_ois_df["benchMarkTrendData"].apply(pd.Series)],
    axis=1
)

mibor_ois_df = pd.concat(
    [mibor_ois_df.drop(columns="benchMarkTrend"),
     mibor_ois_df['benchMarkTrend'].apply(pd.Series)],
    axis= 1
    )

#flatten rates data for each date
for i in range(61):
  mibor_ois_df = pd.concat(
      [mibor_ois_df.drop(columns= i),
      mibor_ois_df[i].apply(pd.Series)],
      axis= 1
        )

#stack rate data of different dates into single column
records = []

for i in range(1, mibor_ois_df.shape[1], 2):
  temp = pd.DataFrame({
      "productName": mibor_ois_df.iloc[:, 0],
      "publishDate": mibor_ois_df.iloc[:, i],
      "rate": mibor_ois_df.iloc[:, i + 1]
  })

  records.append(temp)

result = pd.concat(records, ignore_index=True)

result["publishDate"] = pd.to_datetime(result["publishDate"])

#pivot the dataframe into datewise data
final_mibor_ois_df = result.pivot(
    index="publishDate",
    columns="productName",
    values="rate"
).sort_index()

# sort the data in descending order of date
final_mibor_ois_df.sort_index(ascending= False, inplace= True)

### Reading overnight OIS rates

In [4]:
url = "https://www.fbil.org.in/wasdm/ovnmibor/fetchgraphdata/3M"

r = requests.get(url)
r.raise_for_status()

# print(r.status_code)

data = r.json()

on_MIBOR_df = pd.DataFrame(data)

on_MIBOR_df = pd.concat(
    [on_MIBOR_df.drop(columns="benchMarkTrend"),
     on_MIBOR_df["benchMarkTrend"].apply(pd.Series)],
    axis=1
)

on_MIBOR_df = on_MIBOR_df.rename(columns = {'publishDate': 'Date'})

on_MIBOR_df['Date'] = pd.to_datetime(on_MIBOR_df['Date'])
on_MIBOR_df.drop(columns= ['productName'], inplace= True)